# Test nhanh MCoT-MVS trên Kaggle

Bật **Internet**, chọn **GPU T4 x2** và gắn hai Kaggle datasets:

- **mcot-mvs-weights**, với thư mục `/kaggle/input/datasets/b22dckh099lngminhqu/mcot-mvs-weights/weights`.
- **mcot-vectors**, chứa `gallery_dress_vectors.pt`. Nếu Kaggle gắn ở chỗ khác, sửa `GALLERY_PATH` bên dưới.

Chạy các cell từ trên xuống. Segment chạy trên GPU 0 và Qwen2.5-VL-3B FP16 chạy qua vLLM API trên GPU 1. CIR đợi cả hai rồi chấm điểm gallery. Notebook tự tạo credentials tạm cho FastAPI localhost.

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working
wget -q -O setup.sh https://raw.githubusercontent.com/LMQ-Ptit/MCoT-MVS-Online/quylm_v2/setup.sh
BRANCH=quylm_v2 bash setup.sh
bash /kaggle/working/MCoT-MVS-Online/kaggle_vllm.sh install

In [ ]:
from pathlib import Path

REPO = Path("/kaggle/working/MCoT-MVS-Online")
PY310 = "/kaggle/working/miniconda/envs/py310/bin/python"
GALLERY_PATH = Path("/kaggle/input/datasets/thichuy/mcot-vectors/gallery_dress_vectors.pt")

IMAGE_URL = None  # None dùng ảnh váy xanh của The Met. Có thể điền URL ảnh khác.
TEXT = "make it black"
TOP_K = 5
REPEATS = 3
RUN_SEQUENTIAL_COMPARISON = False  # True để restart FastAPI và đo A/B với cùng vLLM.

In [ ]:
import json
import os
import secrets
import signal
import socket
import subprocess
import time
from urllib.error import URLError
from urllib.request import urlopen

VLLM_BASE = "http://127.0.0.1:8001/v1"
SEARCH_BASE = "http://127.0.0.1:8000"
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
search_api_key = secrets.token_urlsafe(32)
jwt_secret = secrets.token_urlsafe(32)


def stop_process(process):
    if process is None or process.poll() is not None:
        return
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass
    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        process.wait()


def start_process(name, command, port, ready_url, *, env=None, model_id=None):
    with socket.socket() as connection:
        connection.settimeout(1)
        if connection.connect_ex(("127.0.0.1", port)) == 0:
            raise RuntimeError(f"Cổng {port} đang bận. Dừng server cũ trước khi chạy cell này.")

    log_path = Path("/kaggle/working") / f"{name}.log"
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            command, cwd=REPO, env=env, stdout=log,
            stderr=subprocess.STDOUT, start_new_session=True,
        )

    try:
        deadline = time.monotonic() + 900
        while time.monotonic() < deadline:
            if process.poll() is not None:
                raise RuntimeError(f"{name} đã dừng với mã {process.returncode}.")
            try:
                with urlopen(ready_url, timeout=2) as response:
                    ready = model_id is None or any(
                        model["id"] == model_id for model in json.load(response)["data"]
                    )
                if ready:
                    print(f"{name} đã sẵn sàng | log: {log_path}")
                    return process
            except (URLError, TimeoutError):
                pass
            time.sleep(2)
        raise TimeoutError(f"{name} chưa sẵn sàng sau 15 phút.")
    except BaseException:
        stop_process(process)
        print("\n".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-30:]))
        raise


def start_search(parallel=True):
    env = {
        **os.environ,
        "PYTHONPATH": "",
        "MCOT_SEARCH_API_KEY": search_api_key,
        "MCOT_SEARCH_JWT_SECRET": jwt_secret,
        "MCOT_MODEL_NAME": MODEL_ID,
        "MCOT_EXTRACTOR_MODE": "api",
        "MCOT_PARALLEL_BRANCHES": "1" if parallel else "0",
        "MCOT_VLLM_BASE_URL": VLLM_BASE,
        "MCOT_GALLERY_PATH": str(GALLERY_PATH),
    }
    return start_process(
        "mcot-search",
        [PY310, "-m", "uvicorn", "search_api:app", "--app-dir", str(REPO / "src"),
         "--host", "127.0.0.1", "--port", "8000"],
        8000, f"{SEARCH_BASE}/openapi.json", env=env,
    )


if not GALLERY_PATH.is_file():
    raise FileNotFoundError(f"Gắn dataset mcot-vectors hoặc sửa GALLERY_PATH: {GALLERY_PATH}")

stop_process(globals().get("search_process"))
stop_process(globals().get("vllm_process"))
vllm_process = start_process(
    "mcot-vllm", ["bash", str(REPO / "kaggle_vllm.sh"), "serve-pipeline"],
    8001, f"{VLLM_BASE}/models", model_id=MODEL_ID,
)
search_process = start_search(parallel=True)

In [ ]:
import sys
from io import BytesIO
from re import findall
from statistics import median
from time import perf_counter

import requests

sys.path.insert(0, str(REPO / "src"))
from benchmark_latency import load_test_image

img = load_test_image(IMAGE_URL) if IMAGE_URL else load_test_image()
with BytesIO() as buffer:
    img.save(buffer, format="PNG")
    image_bytes = buffer.getvalue()


def run_benchmark(label):
    if REPEATS < 1:
        raise ValueError("REPEATS phải ít nhất là 1.")
    token = requests.post(
        f"{SEARCH_BASE}/api/token", headers={"X-API-KEY": search_api_key}, timeout=30,
    )
    token.raise_for_status()
    headers = {"Authorization": f"Bearer {token.json()['access_token']}"}

    def search_once():
        response = requests.post(
            f"{SEARCH_BASE}/api/list", headers=headers,
            files={"image": ("query.png", image_bytes, "image/png")},
            data={"text": TEXT, "top_k": TOP_K}, timeout=120,
        )
        response.raise_for_status()
        return response

    print(f"{label} | ảnh {img.width}×{img.height} | prompt: {TEXT}")
    search_once()  # Warmup toàn pipeline.
    samples = []
    for trial in range(1, REPEATS + 1):
        started = perf_counter()
        response = search_once()
        client_ms = (perf_counter() - started) * 1000
        header = response.headers["Server-Timing"]
        timings = {key: float(value) for key, value in findall(r"(\w+);dur=([\d.]+)", header)}
        samples.append({"client": client_ms, **timings})
        print(f"Lượt {trial}: client={client_ms:.0f} ms | {header}")

    medians = {key: median(sample[key] for sample in samples) for key in samples[0]}
    print("Latency trung vị (ms):", {key: round(value, 1) for key, value in medians.items()})
    print("Top-K lượt cuối:", response.json()["results"])
    return medians


parallel_timings = run_benchmark("Song song")

In [ ]:
if RUN_SEQUENTIAL_COMPARISON:
    # Giữ vLLM đang chạy. Sau phép đo, FastAPI đang ở chế độ tuần tự.
    stop_process(search_process)
    search_process = start_search(parallel=False)
    sequential_timings = run_benchmark("Tuần tự")
    for key in ("client", "branches", "total"):
        baseline = sequential_timings[key]
        saved_ms = baseline - parallel_timings[key]
        print(f"{key}: tuần tự {baseline:.1f} → song song {parallel_timings[key]:.1f} ms "
              f"| giảm {saved_ms:.1f} ms ({100 * saved_ms / baseline:.1f}%)")

`client` là thời gian round trip của `/api/list`. `branches` là thời gian từ lúc bắt đầu Segment/Qwen tới khi cả hai xong. `total` là thời gian xử lý trong FastAPI. Đọc các giá trị theo mili giây.

Để giải phóng GPU, chạy `stop_process(search_process)` rồi `stop_process(vllm_process)` trong một cell. Để đo A/B, đặt `RUN_SEQUENTIAL_COMPARISON = True` ở cell cấu hình trước khi chạy notebook.